# Bài tập cvxpy có tự chấm

Điền vào các chỗ `TODO` (xoá `raise NotImplementedError`), chạy ô của hàm, rồi chạy ô **Chấm bài** ở cuối. Mỗi hàm có đặc tả đầu vào/đầu ra trong docstring. Test so sánh với đáp án đã được đối chiếu bằng `scipy` (cách giải khác).

In [ ]:
try:
    import cvxpy as cp
except ImportError:
    %pip install -q cvxpy
    import cvxpy as cp
import numpy as np
from scipy.optimize import minimize, nnls
print('cvxpy', cp.__version__)

## Bài 1: `kkt_multipliers_example1`

In [ ]:
def kkt_multipliers_example1():
    """Ví dụ 1 slide 2: min (x-1)^2 + y - 2  s.t.  x + y - 2 <= 0,  x - y + 1 = 0.

    Đầu ra: dict {"x","y","f","lambda","mu"} (float). Gợi ý: constraint.dual_value là nhân tử.
    """
    # TODO
    raise NotImplementedError


## Bài 2: `min_variance_portfolio`

In [ ]:
def min_variance_portfolio(Sigma):
    """min w' Sigma w  s.t.  sum(w) = 1, w >= 0.

    Đầu vào: Sigma (list[list[float]]). Đầu ra: dict {"w": list[float], "variance": float}.
    Gợi ý: cp.quad_form(w, cp.psd_wrap(S)).
    """
    # TODO
    raise NotImplementedError


## Bài 3: `project_onto_polyhedron`

In [ ]:
def project_onto_polyhedron(point, A, b):
    """Điểm gần nhất với point trong {x : A x <= b}.

    Đầu vào: point (list[float]), A (m x n), b (m). Đầu ra: dict {"x": list[float], "distance_sq": float}.
    Gợi ý: cp.sum_squares(x - point).
    """
    # TODO
    raise NotImplementedError


## Bài 4: `nonneg_least_squares`

In [ ]:
def nonneg_least_squares(A, b):
    """min ||A x - b||^2  s.t.  x >= 0.

    Đầu vào: A (m x n), b (m). Đầu ra: dict {"x": list[float], "residual_sq": float}.
    """
    # TODO
    raise NotImplementedError


## Chấm bài

In [ ]:
import types, sys, math
class _P:
    @staticmethod
    def approx(v, rel=1e-6, abs=1e-12):
        return _Approx(v, rel, abs)
class _Approx:
    def __init__(s, v, rel, abs): s.v, s.rel, s.abs = v, rel, abs
    def __eq__(s, o):
        a, b = np.array(o, float), np.array(s.v, float)
        return bool(np.all(np.abs(a - b) <= np.maximum(s.abs, s.rel * np.abs(b))))
pytest = _P
mod = types.SimpleNamespace(**{k: v for k, v in globals().items() if callable(v) and not k.startswith('_')})

SIGMA = [[0.01, 0.0120, 0.0040], [0.0120, 0.0225, 0.0270], [0.0040, 0.0270, 0.0400]]   # case study Module 2 (σ = 10%, 15%, 20%)
A_POLY = [[1, 1], [3, 1], [-1, 0], [0, -1]]; B_POLY = [1, 1.5, 0, 0]


def test_kkt_multipliers_example1():
    r = mod.kkt_multipliers_example1()
    assert r["x"] == pytest.approx(0.5, abs=1e-4) and r["y"] == pytest.approx(1.5, abs=1e-4)
    assert r["f"] == pytest.approx(-0.25, abs=1e-5)
    assert r["lambda"] == pytest.approx(0.0, abs=1e-4) and r["mu"] == pytest.approx(1.0, abs=1e-4)


def test_min_variance_portfolio():
    r = mod.min_variance_portfolio(SIGMA)
    w = np.array(r["w"])
    assert abs(w.sum() - 1) < 1e-6 and w.min() >= -1e-6                      # ràng buộc thỏa
    S = np.array(SIGMA)
    ref = minimize(lambda v: v @ S @ v, np.ones(3) / 3, bounds=[(0, 1)] * 3, constraints=[{"type": "eq", "fun": lambda v: v.sum() - 1}], method="SLSQP", options={"ftol": 1e-14})
    assert r["variance"] == pytest.approx(ref.fun, rel=1e-4, abs=1e-8)       # đối chiếu scipy (cách giải khác)
    assert np.allclose(w, ref.x, atol=1e-3)


def test_project_onto_polyhedron_slide3():
    r = mod.project_onto_polyhedron([1, 0.5], A_POLY, B_POLY)
    assert r["x"] == pytest.approx([0.4, 0.3], abs=1e-4)                    # ví dụ slide 3: x* = (0.4, 0.3)
    assert r["distance_sq"] == pytest.approx(0.4, abs=1e-5)


def test_project_inside_point_is_itself():
    r = mod.project_onto_polyhedron([0.1, 0.1], A_POLY, B_POLY)
    assert r["x"] == pytest.approx([0.1, 0.1], abs=1e-5) and r["distance_sq"] == pytest.approx(0.0, abs=1e-8)


def test_nonneg_least_squares():
    rng = np.random.default_rng(7)
    A = rng.normal(size=(8, 4)); b = rng.normal(size=8)
    r = mod.nonneg_least_squares(A.tolist(), b.tolist())
    x_ref, rn = nnls(A, b)                                                  # đối chiếu scipy.optimize.nnls
    assert min(r["x"]) >= -1e-9
    assert np.allclose(r["x"], x_ref, atol=1e-4) and r["residual_sq"] == pytest.approx(rn ** 2, rel=1e-4, abs=1e-8)


def chấm():
    ok = 0
    tests = [(k, v) for k, v in globals().items() if k.startswith('test_') and callable(v)]
    for k, v in tests:
        try:
            v(); print('✔', k); ok += 1
        except NotImplementedError:
            print('… chưa làm:', k)
        except AssertionError as e:
            print('✘', k, '— sai kết quả')
        except Exception as e:
            print('✘', k, '— lỗi:', type(e).__name__, e)
    print(f'\nĐạt {ok}/{len(tests)} test')
chấm()
